# Lab 06 — Supervisor Routing and Sub-Agent Delegation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tertiarycourses/TGS-2020503207-AI-Vibe-Coding-for-Multi-Agents-System/blob/main/labs/lab-06-supervisor-routing-and-sub-agent-delegation/lab-06-supervisor-routing-and-sub-agent-delegation.ipynb)

**Topic:** 3 — Multi-Agent System Development with OpenAI Agents SDK

**Objective:** Construct a collaborative multi-agent system with supervisor routing and handoffs

Move from one overloaded agent to a team. Build three specialists and a triage supervisor that classifies each request and hands it to the right one.

Full step-by-step instructions are in the Learner Guide.


In [ ]:
!pip install -q openai-agents pydantic python-dotenv


In [ ]:
# API keys: prefer Colab Secrets (key icon in the left sidebar).
# Add each secret there, enable notebook access, then run this cell.
# NEVER paste a key into the notebook - a saved notebook keeps it forever.
import os
from getpass import getpass

try:
    from google.colab import userdata  # available in Colab only
except ImportError:
    userdata = None


def set_key(name: str) -> None:
    """Read a secret from Colab Secrets, falling back to a hidden prompt."""
    if os.environ.get(name):
        return
    value = None
    if userdata is not None:
        try:
            value = userdata.get(name)
        except Exception:
            value = None
    if not value:
        value = getpass(f"Enter {name}: ")
    os.environ[name] = value


set_key("OPENAI_API_KEY")
print("Keys set:", [v for v in ["OPENAI_API_KEY"] if os.environ.get(v)])


## 1. Three narrow tools

The discipline here is *narrowness*: each agent's instructions describe one job, and it holds only the tools that job needs. An agent given every tool behaves like the overloaded single agent you are moving away from.


In [ ]:
from agents import Agent, Runner, function_tool

MODEL = "gpt-4o-mini"


@function_tool
def search_notes(query: str) -> str:
    """Search the internal knowledge base for background on a topic.

    Use this when answering a factual or research question.

    Args:
        query: The search terms.
    """
    notes = {
        "mrt": "The Thomson-East Coast Line opened in stages from 2020.",
        "python": "Python 3.11 introduced significant interpreter speedups.",
        "agents": "Multi-agent systems split work across specialised agents.",
    }
    hits = [text for key, text in notes.items() if key in query.lower()]
    return "\n".join(hits) if hits else "No relevant notes found."


@function_tool
def run_linter(code: str) -> str:
    """Check a Python snippet for syntax errors before returning it.

    Use this to verify any code you are about to give the user.

    Args:
        code: The Python source to check.
    """
    try:
        compile(code, "<submitted>", "exec")
    except SyntaxError as exc:
        return f"Syntax error on line {exc.lineno}: {exc.msg}"
    return "Syntax OK."


@function_tool
def count_words(text: str) -> str:
    """Count the words in a draft, to check it meets a length requirement.

    Args:
        text: The draft text.
    """
    return f"{len(text.split())} words."


## 2. Three specialist agents

`handoff_description` is the field the triage agent reads when deciding where to route. It is to agents what a tool docstring is to tools — vague text here produces bad routing.


In [ ]:
research_agent = Agent(
    name="Research Agent",
    handoff_description="Answers factual and research questions using the knowledge base.",
    instructions=(
        "You are a research specialist. Answer factual questions using the "
        "search_notes tool. Cite what the notes said. If the notes contain "
        "nothing relevant, say so plainly rather than inventing an answer."
    ),
    model=MODEL,
    tools=[search_notes],
)

coding_agent = Agent(
    name="Coding Agent",
    handoff_description="Writes, explains and debugs Python code.",
    instructions=(
        "You are a Python specialist. Write clear, correct code with type hints. "
        "Always check your code with the run_linter tool before returning it, "
        "and fix anything it reports."
    ),
    model=MODEL,
    tools=[run_linter],
)

writing_agent = Agent(
    name="Writing Agent",
    handoff_description="Drafts, edits and summarises prose for a business audience.",
    instructions=(
        "You are a writing specialist. Produce clear, concise prose for a "
        "business audience. Use count_words to confirm you have met any length "
        "requirement the user stated."
    ),
    model=MODEL,
    tools=[count_words],
)


## 3. A guardrail that rejects out-of-scope requests

A guardrail runs *before* the agent does its work, so an out-of-scope request costs one cheap classification instead of a full specialist run. Setting `tripwire_triggered=True` halts the run by raising.


In [ ]:
from pydantic import BaseModel
from agents import (
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    RunContextWrapper,
    input_guardrail,
)


class ScopeCheck(BaseModel):
    is_out_of_scope: bool
    reasoning: str


scope_agent = Agent(
    name="Scope Check",
    instructions=(
        "Decide whether the request is within scope for a team that handles "
        "research, Python coding and business writing. Medical, legal and "
        "financial advice are out of scope. Set is_out_of_scope accordingly."
    ),
    model=MODEL,
    output_type=ScopeCheck,
)


@input_guardrail
async def scope_guardrail(
    ctx: RunContextWrapper[None], agent: Agent, user_input
) -> GuardrailFunctionOutput:
    """Reject out-of-scope requests before any specialist runs."""
    result = await Runner.run(scope_agent, user_input, context=ctx.context)
    check = result.final_output
    return GuardrailFunctionOutput(
        output_info=check,
        tripwire_triggered=check.is_out_of_scope,
    )


## 4. The triage supervisor

A one-line "route appropriately" instruction will misroute. Explicit rules plus a stated default for ambiguity are what make routing reliable.

A **handoff** transfers the conversation: the specialist takes over and produces the final output. That differs from the sub-agents-as-tools pattern in Lab 11, where the parent stays in control and receives the child's answer back.


In [ ]:
triage_agent = Agent(
    name="Triage Agent",
    instructions=(
        "You are a supervisor agent. You never answer the question yourself; "
        "your only job is to route it to exactly one specialist.\n\n"
        "Routing rules:\n"
        "- Facts, background, 'what is', 'when did', research questions "
        "  -> Research Agent.\n"
        "- Anything involving code, programming, debugging, or a library "
        "  -> Coding Agent.\n"
        "- Drafting, editing, summarising, emails, reports "
        "  -> Writing Agent.\n\n"
        "If the request is ambiguous or spans several areas, route to the "
        "Research Agent as the default."
    ),
    model=MODEL,
    handoffs=[research_agent, coding_agent, writing_agent],
    input_guardrails=[scope_guardrail],
)
print("Team ready:", [a.name for a in triage_agent.handoffs])


## 5. Route three requests and confirm each reaches the intended specialist

`result.last_agent` tells you which agent actually produced the answer. Expect Research, Coding and Writing respectively. If a request lands on the wrong specialist, fix the `handoff_description` and the routing rules — not the request.


In [ ]:
REQUESTS = [
    "When did the Thomson-East Coast Line open?",
    "Write a Python function that reverses a linked list.",
    "Draft a 50-word update to my manager about project delays.",
]

for request in REQUESTS:
    result = Runner.run_sync(triage_agent, request)
    print(f"\nRequest:    {request}")
    print(f"Handled by: {result.last_agent.name}")
    print(f"Answer:     {result.final_output[:200]}")


## 6. Confirm the guardrail refuses out-of-scope work

The medical request must raise `InputGuardrailTripwireTriggered` **before** any specialist runs — no specialist span should appear in the trace.


In [ ]:
for request in REQUESTS + ["Should I take ibuprofen for my headache?"]:
    try:
        result = Runner.run_sync(triage_agent, request)
        print(f"\n{request}\n  -> {result.last_agent.name}")
    except InputGuardrailTripwireTriggered as exc:
        reason = exc.guardrail_result.output.output_info.reasoning
        print(f"\n{request}\n  -> REFUSED: {reason}")


## 7. Compare against a single agent given all the tools

Watch for the single agent skipping `run_linter` on a coding request, or answering a factual question from its own weights instead of calling `search_notes`. With one narrow instruction set per specialist, that drift largely disappears — which is the reliability argument for multi-agent systems.

Open <https://platform.openai.com/traces> to see the handoff chain, with duration and token counts per span.


In [ ]:
kitchen_sink_agent = Agent(
    name="Everything Agent",
    instructions=(
        "You handle research, Python coding and business writing. "
        "Use whichever tool is appropriate."
    ),
    model=MODEL,
    tools=[search_notes, run_linter, count_words],
)

for request in REQUESTS:
    single = Runner.run_sync(kitchen_sink_agent, request)
    team = Runner.run_sync(triage_agent, request)
    print(f"\nRequest: {request}")
    print(f"  single-agent tools used: "
          f"{sum(1 for i in single.new_items if 'ToolCall' in type(i).__name__)}")
    print(f"  team specialist:         {team.last_agent.name}")
